# M3 TimeGrad (diffusion) su GPU Colab — run dedicato

Notebook **solo M3**, per una **sessione fresca** (o un secondo account Google) dedicata al run pesante di TimeGrad. M0/M1 restano sul laptop; M2 DeepAR gira nell'altro notebook (`colab_m2_m3.ipynb`).

> ⚠️ **Colab FREE = una sola GPU per account Google.** Se apri questo *in parallelo* al run M2 **sullo stesso account**, Colab puo` rifiutare la seconda GPU **o disconnettere il run M2** (niente resume → perdi il lavoro fatto). Per girare DAVVERO in parallelo serve un **secondo account Google** (GPU indipendente). Altrimenti lancialo **dopo** che M2 ha finito, in una sessione fresca.

Esegui le celle dall'alto. `DATASET = "electricity"` e` il run vero (predict ~2-3h, niente resume: tieni il tab aperto); metti `"exchange"` per un giro di prova veloce.

## 1 - Verifica la GPU
Se vedi una **Tesla T4** (o simile) la GPU e` attiva. Altrimenti: **Runtime > Change runtime type > GPU**, poi riesegui.

In [ ]:
!nvidia-smi

## 2 - Scarica il codice del progetto
Cloniamo il repo sul branch di lavoro `feature/port-ladder-exchange`.

In [ ]:
REPO_URL = "https://github.com/Icaica14/pml-diffusion-tsf.git"
BRANCH   = "feature/port-ladder-exchange"

import os
if not os.path.isdir("/content/pml-diffusion-tsf"):
    !git clone --branch $BRANCH $REPO_URL /content/pml-diffusion-tsf
%cd /content/pml-diffusion-tsf
!git log --oneline -1

## 3 - Installa le librerie pesanti (versioni bloccate)
Stesse versioni del notebook M2/M3: **GluonTS 0.13** + **PyTorchTS** + il PyTorch CUDA gia` presente su Colab.

**Attenzione:** downgrade di **NumPy < 2** e **pandas < 2.2** (servono a GluonTS 0.13). A fine cella: **Runtime > Restart session**, poi **riparti dalla cella 4** (NON rifare questa).

In [ ]:
!pip install -q "gluonts[torch]==0.13.7" "numpy<2" "pandas<2.2"
!pip install -q --no-deps "git+https://github.com/zalandoresearch/pytorch-ts.git@81be06bcc"
print("\nInstallazione finita. Ora: Runtime > Restart session, poi continua dalla cella 4.")

## 4 - Controllo ambiente (dopo il restart)
Se arriva in fondo senza errori e vedi `cuda disponibile: True`, sei pronto. Se esplode, copiami l'output.

In [ ]:
%cd /content/pml-diffusion-tsf
import numpy, pandas, torch, gluonts, pts
print("numpy   :", numpy.__version__)
print("pandas  :", pandas.__version__)
print("torch   :", torch.__version__)
print("gluonts :", gluonts.__version__)
print("cuda disponibile:", torch.cuda.is_available())
print("device  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 5 - Scegli il dataset
`"electricity"` e` il run vero (D=321). `"exchange"` per pratica (veloce). `CHUNK` serve solo a Electricity: valuta i campioni a blocchi per non esaurire la RAM.

In [ ]:
DATASET = "electricity"        # "exchange" per un giro veloce di prova
CONFIG  = f"configs/data_{DATASET}.yaml"
CHUNK   = 256 if DATASET == "electricity" else 0
EPOCHS  = 50 if DATASET == "electricity" else 20
print(f"dataset={DATASET}  config={CONFIG}  chunk={CHUNK}  epochs={EPOCHS}")

## 6 - M3: TimeGrad (diffusion)
Diffusione condizionata + RNN multivariato. Su GPU il pezzo lento e` il **campionamento** dei 100 trajectory per finestra. Su Electricity: run lungo, **niente resume** — tieni il tab aperto. Se la prima volta da` un errore di shape o su `input_size`, copiamelo.

In [ ]:
!python -m experiments.run_timegrad --config $CONFIG --chunk $CHUNK --epochs $EPOCHS --device cuda

## 7 - Porta a casa i risultati
Colab e` effimero. Salviamo **solo le righe TimeGrad** (non toccano M0/M1/M2 in locale), le stampiamo e le scarichiamo. (Se c'e` gia` una riga timegrad Exchange committata esce anche quella: la deduplico io lato laptop.)

In [ ]:
import pathlib
lines = pathlib.Path("results/registry.csv").read_text().splitlines()
header = lines[0]
new = [l for l in lines[1:] if len(l.split(",")) > 2 and l.split(",")[2] == "timegrad"]
out = "\n".join([header] + new) + "\n"
pathlib.Path("results/colab_new_rows.csv").write_text(out)
print(out)
print(f"--> {len(new)} righe timegrad in results/colab_new_rows.csv")

In [ ]:
from google.colab import files
files.download("results/colab_new_rows.csv")

## 8 - E adesso?
Mandami l'output della cella 7 (o caricami `colab_new_rows.csv`): unisco la riga `timegrad` Electricity alla `results/registry.csv` del repo e committo io in locale.